# Tạo gương mặt AI và bộ clip nền cho AI host (spec 005, ADR-036)

Notebook chạy **trên Kaggle hoặc Colab**, không chạy trên laptop. Nó bật ComfyUI không giao diện web, tạo ảnh gương mặt bằng
SDXL base 1.0, rồi tạo các clip nền không tiếng (chờ, nói; cử chỉ để feature sau) bằng Wan 2.2 TI2V-5B. Miệng **không** lấy từ clip: lúc
live, LiveTalking tạo miệng theo thời gian thực (spec 005 FR-006).

**Quy tắc (CLAUDE.md, spec 005 FR-004, FR-005):**
- Gương mặt là người không có thật, trông là người trưởng thành, không giống người nổi tiếng hay người quen. Không đưa ảnh
  người thật vào notebook.
- Mỗi nhân vật chỉ dùng cho một khách, một shop.
- Ảnh, clip và `provenance.json` **không đưa vào git**. Tải về laptop và lưu ở `assets/avatars/<CHARACTER_ID>/`.
- Nhãn "AI" được ghép vào khung hình trong pipeline lúc live, không ghép ở đây.

**Cách chạy trên Kaggle (giai đoạn demo, ADR-036):**
1. kaggle.com → Create → New Notebook → File → Import Notebook → chọn file này.
2. Thanh bên phải, mục Session options: Accelerator = **GPU T4 x2**, Internet = **On** (tài khoản phải xác minh số điện thoại).
3. Sửa ô "Cấu hình", rồi chạy lần lượt từng ô. Ô chọn gương mặt cần sửa `CHOSEN_FACE_INDEX` sau khi xem ảnh.
4. Tải `avatar_out/<CHARACTER_ID>.zip` ở panel Output bên phải (thư mục `/kaggle/working`).

**Colab:** File → Upload notebook, Runtime → Change runtime type → T4 GPU. Colab bản miễn phí chỉ có khoảng 12 GB RAM nên có
thể bị crash lúc nạp model video; khi đó quay lại Kaggle.

**Khi lỗi:** xem `avatar_out/comfyui.log`. Hết VRAM: thêm `"--lowvram"` vào `EXTRA_SERVER_ARGS`. Clip ra toàn màu đen (tràn số
fp16 trên T4): thử `"--fp32-unet"` (chậm hơn nhiều). Ghi lại lỗi và thời gian chạy để điền vào plan 005.

## 1. Cấu hình

Chỉ sửa ô này. Mọi giá trị được ghi vào `provenance.json`.

In [ ]:
CHARACTER_ID = "demo_host_01"  # one character = one customer, one shop
CREATED_BY = ""  # who runs this notebook; required in provenance

# Spec 005 chose waist-up framing (clarify 2026-11-05) so later gesture clips can reuse the same character.
FRAMING = "half_body"  # or "head_and_shoulders"
FRAMING_PROMPTS = {
    "head_and_shoulders": "head and shoulders portrait, centered, facing the camera",
    "half_body": "medium shot from the waist up, both hands visible, centered, facing the camera",
}

FACE_PROMPT = (
    "photo of a friendly fictional Vietnamese woman in her late twenties, livestream sales host, natural makeup, "
    "neat dark hair, simple pastel blouse, warm smile with lips closed, looking straight at the camera, "
    "soft studio lighting, plain light beige background, sharp focus, realistic skin texture"
)
FACE_NEGATIVE = (
    "child, teenager, young-looking, celebrity, famous person, cartoon, anime, 3d render, text, watermark, logo, "
    "open mouth, teeth, hands near face, sunglasses, hat, busy background, blurry, deformed, extra fingers"
)
FACE_SEEDS = [11, 22, 33, 44]
FACE_SIZE = (832, 1216)  # SDXL portrait bucket (width, height)

# Wan 2.2 TI2V-5B: width and height are multiples of 32, length is 4n + 1 frames at 24 fps.
# The model is trained at 1280x704; 480x832 is smaller so a T4 finishes in reasonable time (to be measured).
CLIP_SIZE = (480, 832)
CLIP_LENGTH = 121  # about 5 seconds
CLIP_FPS = 24
CLIP_STEPS = 20

# The mouth is redrawn live by LiveTalking, so every clip asks for closed lips (spec 005 edge case: a base clip
# whose mouth moves on its own is rejected).
CLIP_NEGATIVE = (
    "talking, speaking, mouth moving, open mouth, teeth, lip movement, camera movement, zoom, cut, scene change, "
    "blurry, distorted face, face morphing, deformed hands, extra fingers, text, subtitles, watermark, flicker"
)
CLIPS = [
    {
        "name": "idle_01",
        "type": "idle",
        "tags": [],
        "seed": 101,
        "prompt": "the woman stands still and waits calmly, gentle breathing, natural blinking, "
        "a very slight head tilt, lips closed, static camera",
    },
    {
        "name": "talk_01",
        "type": "talking",
        "tags": [],
        "seed": 201,
        "prompt": "the woman listens and reacts as if chatting with viewers, small natural head nods, "
        "friendly expression, lips closed, static camera",
    },
    {
        "name": "talk_02",
        "type": "talking",
        "tags": [],
        "seed": 202,
        "prompt": "the woman explains something with calm energy, slight head turns left and right "
        "then back to the camera, lips closed, static camera",
    },
    {
        "name": "idle_02",
        "type": "idle",
        "tags": [],
        "seed": 102,
        "prompt": "the woman waits with a relaxed posture, hands resting together in front, slow breathing, "
        "natural blinking, a slight shift of weight, lips closed, static camera",
    },
    # Gesture clips are deferred to a later feature (spec 005 clarify 2026-11-05); v1 rotates idle and talking clips.
]
# Idle clips get an extra ping-pong copy (forward then backward), which always loops without a jump (spec 005 FR-008).
PINGPONG_TYPES = {"idle"}

EXTRA_SERVER_ARGS: list[str] = []  # e.g. ["--lowvram"] or ["--fp32-unet"], see the notes above

## 2. Môi trường và ComfyUI

In [ ]:
import json
import shutil
import subprocess
import sys
import time
import urllib.error
import urllib.request
from datetime import UTC, datetime
from pathlib import Path

import torch  # preinstalled by Kaggle/Colab with the right CUDA build

ON_KAGGLE = Path("/kaggle").exists()
ON_COLAB = "google.colab" in sys.modules
if not (ON_KAGGLE or ON_COLAB):
    raise SystemExit("Notebook này chỉ chạy trên Kaggle hoặc Colab (ADR-036), không chạy trên laptop.")
PLATFORM = "kaggle" if ON_KAGGLE else "colab"

# Weights stay out of /kaggle/working: that folder is the saved output and is capped at 20 GB.
COMFY_DIR = Path("/tmp/ComfyUI") if ON_KAGGLE else Path("/content/ComfyUI")
OUT_ROOT = (Path("/kaggle/working") if ON_KAGGLE else Path("/content")) / "avatar_out"
OUT_DIR = OUT_ROOT / CHARACTER_ID
OUT_DIR.mkdir(parents=True, exist_ok=True)


def run(*cmd: str, cwd: Path | None = None) -> str:
    result = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"{' '.join(cmd[:4])} lỗi:\n{result.stderr[-3000:]}")
    return result.stdout.strip()


if not COMFY_DIR.exists():
    run("git", "clone", "--depth", "1", "https://github.com/comfyanonymous/ComfyUI", str(COMFY_DIR))
COMFY_COMMIT = run("git", "rev-parse", "HEAD", cwd=COMFY_DIR)

# Keep the platform's CUDA build of torch: reinstalling it from PyPI can break the GPU runtime.
SKIP = {"torch", "torchvision", "torchaudio"}
reqs = []
for line in (COMFY_DIR / "requirements.txt").read_text().splitlines():
    line = line.strip()
    name = line.split(";")[0].split("[")[0]
    for sep in "<>=!~ ":
        name = name.split(sep)[0]
    if line and not line.startswith("#") and name.lower() not in SKIP:
        reqs.append(line)
run(sys.executable, "-m", "pip", "install", "-q", *reqs, "huggingface_hub")

if shutil.which("ffmpeg") is None:
    run("apt-get", "install", "-y", "-qq", "ffmpeg")

GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
print(f"Nền tảng: {PLATFORM} · GPU: {GPU_NAME} · torch {torch.__version__} · ComfyUI {COMFY_COMMIT[:10]}")
if GPU_NAME == "none":
    raise SystemExit("Chưa bật GPU. Kaggle: Accelerator = GPU T4 x2. Colab: Runtime type = T4 GPU.")

## 3. Tải model (chỉ trong máy Kaggle/Colab, không về laptop)

Khoảng 25 GB. License ghi vào hồ sơ nguồn gốc; TODO (RISKS L10): đọc model card của đúng bản checkpoint trước khi khách trả tiền.

In [ ]:
from huggingface_hub import hf_hub_download

MODELS = {
    "face": {
        "repo": "stabilityai/stable-diffusion-xl-base-1.0",
        "file": "sd_xl_base_1.0.safetensors",
        "dir": "checkpoints",
        "license": "CreativeML Open RAIL++-M",
    },
    "video": {
        "repo": "Comfy-Org/Wan_2.2_ComfyUI_Repackaged",
        "file": "split_files/diffusion_models/wan2.2_ti2v_5B_fp16.safetensors",
        "dir": "diffusion_models",
        "license": "Apache-2.0 (Wan 2.2)",
    },
    "video_vae": {
        "repo": "Comfy-Org/Wan_2.2_ComfyUI_Repackaged",
        "file": "split_files/vae/wan2.2_vae.safetensors",
        "dir": "vae",
        "license": "Apache-2.0 (Wan 2.2)",
    },
    "video_text_encoder": {
        "repo": "Comfy-Org/Wan_2.1_ComfyUI_repackaged",
        "file": "split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors",
        "dir": "text_encoders",
        "license": "TODO: xác nhận (umT5-XXL, đóng gói lại từ Wan 2.1, Apache-2.0)",
    },
}

for key, model in MODELS.items():
    started = time.monotonic()
    cached = Path(hf_hub_download(model["repo"], model["file"]))
    parts = cached.parts
    model["revision"] = parts[parts.index("snapshots") + 1] if "snapshots" in parts else "unknown"
    target = COMFY_DIR / "models" / model["dir"] / cached.name
    target.parent.mkdir(parents=True, exist_ok=True)
    if not target.exists():
        target.symlink_to(cached)
    print(f"{key}: {cached.name} @ {model['revision'][:10]} ({time.monotonic() - started:.0f} s)")

## 4. Bật ComfyUI (chỉ nghe trong máy, không mở giao diện web)

In [ ]:
SERVER = "http://127.0.0.1:8188"
# Stays open while the server runs; the notebook process owns it.
server_log = open(OUT_ROOT / "comfyui.log", "w")
server = subprocess.Popen(
    [
        sys.executable,
        "main.py",
        "--listen",
        "127.0.0.1",
        "--port",
        "8188",
        "--preview-method",
        "none",
        *EXTRA_SERVER_ARGS,
    ],
    cwd=COMFY_DIR,
    stdout=server_log,
    stderr=subprocess.STDOUT,
)


def api(path: str, payload: dict | None = None) -> dict:
    data = None if payload is None else json.dumps(payload).encode()
    request = urllib.request.Request(SERVER + path, data=data, headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(request, timeout=30) as response:
        body = response.read()
    return json.loads(body) if body else {}


def log_tail() -> str:
    return (OUT_ROOT / "comfyui.log").read_text(errors="replace")[-3000:]


for _ in range(120):
    if server.poll() is not None:
        raise RuntimeError("ComfyUI dừng khi khởi động:\n" + log_tail())
    try:
        api("/system_stats")
        break
    except OSError:
        time.sleep(2)
else:
    raise RuntimeError("ComfyUI không lên sau 4 phút:\n" + log_tail())
print("ComfyUI đã chạy.")


def run_workflow(workflow: dict, timeout_s: float) -> dict:
    """Queue one API-format workflow and wait for its outputs."""
    try:
        prompt_id = api("/prompt", {"prompt": workflow})["prompt_id"]
    except urllib.error.HTTPError as err:
        # ComfyUI explains a rejected workflow (missing node, wrong input) in the body.
        raise RuntimeError(err.read().decode(errors="replace")[:3000]) from err
    deadline = time.monotonic() + timeout_s
    while time.monotonic() < deadline:
        entry = api(f"/history/{prompt_id}").get(prompt_id)
        if entry is not None:
            if entry.get("status", {}).get("status_str") == "error":
                raise RuntimeError(json.dumps(entry["status"], ensure_ascii=False)[:3000])
            return entry["outputs"]
        if server.poll() is not None:
            raise RuntimeError("ComfyUI đã dừng giữa chừng:\n" + log_tail())
        time.sleep(5)
    raise TimeoutError(f"Quá {timeout_s:.0f} s mà workflow chưa xong")


def output_images(outputs: dict) -> list[Path]:
    files = []
    for node_output in outputs.values():
        for image in node_output.get("images", []):
            files.append(COMFY_DIR / "output" / image["subfolder"] / image["filename"])
    return files

## 5. Tạo ảnh gương mặt (SDXL)

Xem các ảnh rồi chọn một ở ô sau. Bỏ ảnh trông trẻ con, giống người nổi tiếng hay người quen, hoặc méo mặt (spec 005 FR-005).

In [ ]:
from IPython.display import Image, Video, display


def face_workflow(seed: int, prefix: str) -> dict:
    width, height = FACE_SIZE
    prompt = f"{FACE_PROMPT}, {FRAMING_PROMPTS[FRAMING]}"
    return {
        "1": {"class_type": "CheckpointLoaderSimple", "inputs": {"ckpt_name": "sd_xl_base_1.0.safetensors"}},
        "2": {"class_type": "CLIPTextEncode", "inputs": {"text": prompt, "clip": ["1", 1]}},
        "3": {"class_type": "CLIPTextEncode", "inputs": {"text": FACE_NEGATIVE, "clip": ["1", 1]}},
        "4": {"class_type": "EmptyLatentImage", "inputs": {"width": width, "height": height, "batch_size": 1}},
        "5": {
            "class_type": "KSampler",
            "inputs": {
                "model": ["1", 0],
                "positive": ["2", 0],
                "negative": ["3", 0],
                "latent_image": ["4", 0],
                "seed": seed,
                "steps": 30,
                "cfg": 6.5,
                "sampler_name": "dpmpp_2m",
                "scheduler": "karras",
                "denoise": 1.0,
            },
        },
        "6": {"class_type": "VAEDecode", "inputs": {"samples": ["5", 0], "vae": ["1", 2]}},
        "7": {"class_type": "SaveImage", "inputs": {"images": ["6", 0], "filename_prefix": prefix}},
    }


FACES_DIR = OUT_DIR / "faces"
FACES_DIR.mkdir(exist_ok=True)
faces = []
for index, seed in enumerate(FACE_SEEDS):
    started = time.monotonic()
    [image] = output_images(run_workflow(face_workflow(seed, f"{CHARACTER_ID}_face"), timeout_s=900))
    face_path = FACES_DIR / f"face_{index}_seed{seed}.png"
    shutil.copy(image, face_path)
    seconds = round(time.monotonic() - started, 1)
    faces.append({"index": index, "seed": seed, "file": f"faces/{face_path.name}", "seconds": seconds})
    print(f"Ảnh {index} (seed {seed}): {seconds} s")
    display(Image(filename=str(face_path), width=256))

## 6. Chọn gương mặt

In [ ]:
CHOSEN_FACE_INDEX = 0  # set after looking at the images above

chosen_face = faces[CHOSEN_FACE_INDEX]
shutil.copy(OUT_DIR / chosen_face["file"], COMFY_DIR / "input" / "face.png")
shutil.copy(OUT_DIR / chosen_face["file"], OUT_DIR / "face.png")
# Unload SDXL so the video model has the whole GPU and RAM.
api("/free", {"unload_models": True, "free_memory": True})
display(Image(filename=str(OUT_DIR / "face.png"), width=256))

## 7. Tạo clip nền (Wan 2.2 TI2V-5B)

Mỗi clip in ra thời gian chạy (dùng để điền TODO đo tốc độ T4 trong ADR-036). Chạy lại ô này sẽ bỏ qua clip đã có.

In [ ]:
def clip_workflow(clip: dict, prefix: str) -> dict:
    width, height = CLIP_SIZE
    return {
        "1": {
            "class_type": "UNETLoader",
            "inputs": {"unet_name": Path(MODELS["video"]["file"]).name, "weight_dtype": "default"},
        },
        "2": {
            "class_type": "CLIPLoader",
            "inputs": {"clip_name": Path(MODELS["video_text_encoder"]["file"]).name, "type": "wan"},
        },
        "3": {"class_type": "VAELoader", "inputs": {"vae_name": Path(MODELS["video_vae"]["file"]).name}},
        "4": {"class_type": "ModelSamplingSD3", "inputs": {"model": ["1", 0], "shift": 8.0}},
        "5": {"class_type": "CLIPTextEncode", "inputs": {"text": clip["prompt"], "clip": ["2", 0]}},
        "6": {"class_type": "CLIPTextEncode", "inputs": {"text": CLIP_NEGATIVE, "clip": ["2", 0]}},
        "7": {"class_type": "LoadImage", "inputs": {"image": "face.png"}},
        "8": {
            "class_type": "Wan22ImageToVideoLatent",
            "inputs": {
                "vae": ["3", 0],
                "width": width,
                "height": height,
                "length": CLIP_LENGTH,
                "batch_size": 1,
                "start_image": ["7", 0],
            },
        },
        "9": {
            "class_type": "KSampler",
            "inputs": {
                "model": ["4", 0],
                "positive": ["5", 0],
                "negative": ["6", 0],
                "latent_image": ["8", 0],
                "seed": clip["seed"],
                "steps": CLIP_STEPS,
                "cfg": 5.0,
                "sampler_name": "uni_pc",
                "scheduler": "simple",
                "denoise": 1.0,
            },
        },
        "10": {"class_type": "VAEDecode", "inputs": {"samples": ["9", 0], "vae": ["3", 0]}},
        # Frames are saved as PNG and encoded with ffmpeg below, so the result does not depend on ComfyUI's video nodes.
        "11": {"class_type": "SaveImage", "inputs": {"images": ["10", 0], "filename_prefix": prefix}},
    }


def encode(frames: list[Path], target: Path) -> None:
    staging = OUT_ROOT / "_frames"
    shutil.rmtree(staging, ignore_errors=True)
    staging.mkdir()
    for number, frame in enumerate(frames):
        shutil.copy(frame, staging / f"{number:05d}.png")
    run(
        "ffmpeg", "-y", "-loglevel", "error", "-framerate", str(CLIP_FPS), "-i", str(staging / "%05d.png"),
        "-c:v", "libx264", "-pix_fmt", "yuv420p", "-crf", "16", "-an", str(target),
    )  # fmt: skip
    shutil.rmtree(staging)


CLIPS_DIR = OUT_DIR / "clips"
CLIPS_DIR.mkdir(exist_ok=True)
# Kept across reruns of this cell, so finished clips are not generated again.
if "clip_records" not in globals():
    clip_records = {}
for clip in CLIPS:
    target = CLIPS_DIR / f"{clip['name']}.mp4"
    if target.exists() and clip["name"] in clip_records:
        print(f"{clip['name']}: đã có, bỏ qua")
        continue
    started = time.monotonic()
    frames = output_images(run_workflow(clip_workflow(clip, f"{CHARACTER_ID}_{clip['name']}"), timeout_s=3 * 3600))
    seconds = round(time.monotonic() - started, 1)
    encode(frames, target)
    files = [f"clips/{target.name}"]
    if clip["type"] in PINGPONG_TYPES:
        pingpong = CLIPS_DIR / f"{clip['name']}_pingpong.mp4"
        encode(frames + frames[-2:0:-1], pingpong)
        files.append(f"clips/{pingpong.name}")
    for frame in frames:
        frame.unlink()  # 100+ PNGs per clip would fill the disk
    clip_records[clip["name"]] = {**clip, "files": files, "frames": len(frames), "seconds": seconds}
    print(f"{clip['name']}: {len(frames)} frame, {seconds} s ({seconds / 60:.1f} phút)")
    display(Video(str(target), embed=True, width=240))

## 8. Duyệt và đóng gói

Xem lại từng clip. Bỏ clip có miệng tự cử động, mặt biến dạng, hoặc nhân vật đổi khác so với ảnh (spec 005). Người chạy notebook
phải tự xác nhận hai ý trong `FACE_REVIEW` thì mới đóng gói được.

In [ ]:
FACE_REVIEW = {
    "looks_adult": None,  # True after checking the face looks like an adult
    "resembles_no_real_person": None,  # True after checking it does not look like a celebrity or someone you know
}
REJECTED_CLIPS: list[str] = []  # clip names to leave out, e.g. ["talk_02"]

if not CREATED_BY:
    raise SystemExit("Điền CREATED_BY ở ô Cấu hình rồi chạy lại ô đó và ô này.")
if not all(value is True for value in FACE_REVIEW.values()):
    raise SystemExit("Chưa xác nhận FACE_REVIEW: trông là người trưởng thành, không giống người thật (FR-005).")

for name in REJECTED_CLIPS:
    for file in clip_records[name]["files"]:
        (OUT_DIR / file).unlink(missing_ok=True)

provenance = {
    "character_id": CHARACTER_ID,
    "created_by": CREATED_BY,
    "created_at": datetime.now(UTC).isoformat(timespec="seconds"),
    "kind": "ai_generated_fictional_face",
    "platform": PLATFORM,
    "platform_terms_note": "ADR-036: dùng cho giai đoạn demo; đọc lại điều khoản trước khi khách trả tiền",
    "gpu": GPU_NAME,
    "torch": torch.__version__,
    "comfyui_commit": COMFY_COMMIT,
    "models": MODELS,
    "face": {
        "prompt": f"{FACE_PROMPT}, {FRAMING_PROMPTS[FRAMING]}",
        "negative": FACE_NEGATIVE,
        "size": FACE_SIZE,
        "framing": FRAMING,
        "candidates": faces,
        "chosen_index": CHOSEN_FACE_INDEX,
        "file": "face.png",
        "review": FACE_REVIEW,
    },
    "clip_settings": {
        "size": CLIP_SIZE,
        "length": CLIP_LENGTH,
        "fps": CLIP_FPS,
        "steps": CLIP_STEPS,
        "negative": CLIP_NEGATIVE,
    },
    "clips": [record for name, record in clip_records.items() if name not in REJECTED_CLIPS],
    "rejected_clips": REJECTED_CLIPS,
}
(OUT_DIR / "provenance.json").write_text(json.dumps(provenance, ensure_ascii=False, indent=2), encoding="utf-8")

archive = shutil.make_archive(str(OUT_ROOT / CHARACTER_ID), "zip", root_dir=OUT_ROOT, base_dir=CHARACTER_ID)
print(f"Đã đóng gói: {archive}")
print(f"Giải nén trên laptop vào assets/avatars/ (đã gitignore), không commit. Thư mục: assets/avatars/{CHARACTER_ID}/")
if ON_COLAB:
    from google.colab import files

    files.download(archive)